# Chạy và so sánh hoàn toàn trên Kaggle

Làm theo thứ tự dưới đây. Toàn bộ code trong tài liệu chạy ở **Code cell của Kaggle**. Mỗi notebook GPU chỉ cần gắn **một Input là kit**; nó tự cài thư viện, tải model rồi chạy. Không cần notebook chuẩn bị, Dataset model hay bộ thư viện riêng.

Thời gian tải/cài được ghi trong log, **không cộng vào thời gian so sánh hiệu năng**. Phép đo bắt đầu khi gọi pipeline: báo riêng thời gian load và thời gian xử lý câu hỏi. Vòng agent và logic nghiệp vụ giữ nguyên.

## 1. Upload bộ chạy một lần

Trên Kaggle, tạo **Dataset private** rồi upload **một file `kaggle-runtime-kit.zip`** đã chuẩn bị ở `outputs/packages/` trong dự án. Thao tác bằng trình duyệt, không cần gõ lệnh. Nếu đã có Dataset kit, cập nhật chính Dataset đó bằng file ZIP mới, rồi chọn phiên bản mới trong hai notebook GPU.

Kaggle giải nén archive khi upload Dataset, giữ cấu trúc thư mục. Vì vậy kit mới chứa thẳng `current/` (source mới), `baseline/` (source cũ), `benchmark/` (dataset/IDs) và `kaggle_kit.json`; **không chứa ZIP lồng và không để hai source đè lên nhau**. Code tìm manifest thư mục, đồng thời nhận được cả trường hợp Input vẫn giữ file ZIP. Không phụ thuộc tên thư mục mà Kaggle tạo. [Tài liệu upload Kaggle](https://www.kaggle.com/docs/datasets).

Nếu cập nhật dataset đã có, tạo phiên bản mới và chọn phiên bản đó ở các notebook dưới. Chỉ gắn một phiên bản kit. Hai ZIP source riêng vẫn được giữ để kiểm tra, nhưng không phải upload chúng cho quy trình này.

Lộ trình cần theo:

| Thứ tự | Làm gì? | Có cần canh không? | Khi nào đi tiếp? |
|---|---|---|---|
| 2 | Tạo hai notebook GPU, gắn kit | Chỉ thao tác cấu hình; chưa chạy | Hai notebook có cùng phiên bản kit, T4 × 2 và Internet bật |
| 3 | Chạy thử 4 câu, GPU | **Có**, đây là lượt phát hiện lỗi | Cả Torch và vLLM đều có pilot status=passed |
| 4 | Chạy chính 24 câu, GPU | Sau khi Submit Save & Run All có thể rời máy | Hai version chạy xong; đủ 24 câu thì đọc được báo cáo cả khi needs_review |
| 5 | Mở kết quả hai version | Chỉ kiểm status và tên file | Có đủ hai main_result.zip |
| 6 | So sánh, CPU | Chạy ngắn; không cần GPU | Đọc comparison.txt / comparison.csv |

**Pilot và main là hai lượt khác nhau.** Main không chạy lại bốn câu pilot. Save & Run All tạo phiên sạch, tự cài thư viện/tải model/đọc Input/đo context trước khi chạy; không dùng các biến hay venv còn sót từ lượt tương tác. Quick Save không chạy lại notebook. Đây là hai chế độ khác nhau trong [hướng dẫn chính thức Kaggle](https://www.kaggle.com/docs/notebooks).

## 2. Tạo hai notebook GPU, không chuẩn bị tài nguyên riêng

Tạo notebook **hai-torch-runtime**:

1. Add Input **dataset kit ở bước 1**, chọn phiên bản mới nhất vừa upload. Không cần gắn Output của notebook prepare_model cũ.
2. Chọn Accelerator **GPU T4 × 2**, bật **Internet**.
3. Import `notebooks/DataMate_Kaggle_Runtime_Benchmark.ipynb`, hoặc copy hai cell ở bước 3.
4. Chưa Save & Run All ngay; chạy pilot ở bước 3 trước để phát hiện lỗi.

Tạo notebook **hai-vllm-runtime** tương tự, cùng phiên bản kit, cùng T4 × 2 và Internet bật. Trong cell cấu hình của notebook này đổi `ARM = 'vllm'`. Hai notebook dùng hai phiên GPU riêng; không chạy hai engine tranh cùng hai card trong một notebook.

Nếu đã có notebook/output **hai-model-prepare**, giữ lại tùy nhu cầu khác; quy trình kiểm thử này không dùng đến và không cần tải lại nó thành Dataset. Bản kit mới không còn bước chuẩn bị CPU. Không gắn kit cũ và mới cùng lúc.



## 6. So sánh ngay trên Kaggle bằng CPU

**Không chạy file log.** `.log` là nhật ký mở ra đọc khi có lỗi. File `.zip` kết quả chứa các số đo để chương trình so sánh đọc tự động. Sau hai lượt main, việc tiếp theo là chạy **notebook so sánh CPU**, không phải chạy lại agent hay chép log vào code.

Tạo notebook thứ ba tên **hai-runtime-compare**:

1. Accelerator **None**; không cần GPU.
2. Add Input dataset kit ở bước 1, chọn phiên bản mới.
3. Add Input **Output của version main đã hoàn tất** từ `hai-torch-runtime` và `hai-vllm-runtime`. Kiểm trong Input thấy `torch_main_result.zip` và `vllm_main_result.zip`.
4. Import `notebooks/DataMate_Kaggle_Compare.ipynb`, hoặc copy một cell dưới và chạy. Code tự tìm/giải nén đúng hai file; không cần đoán đường dẫn mount của Kaggle.

<!-- notebook:compare:start -->


In [ ]:
TASK = 'compare'
OPTIONS = {}  # so timing; chưa đọc nhãn/answers

import hashlib, json, runpy, shutil, traceback, zipfile
from pathlib import Path

try:
    inputs = Path(OPTIONS.get('input_root', '/kaggle/input'))
    bootstrap = Path(OPTIONS.get('bootstrap_root', '/tmp/datamate-bootstrap'))
    manifests = list(inputs.rglob('kaggle_kit.json'))
    if not manifests:
        packets = list(inputs.rglob('kaggle-runtime-kit.zip'))
        assert len(packets) == 1, 'Add Input cần đúng một bộ kaggle-runtime-kit mới'
        with packets[0].open('rb') as handle:
            packet_hash = hashlib.file_digest(handle, 'sha256').hexdigest()
        expanded = bootstrap / 'upload' / packet_hash[:16]
        expanded.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(packets[0]) as archive:
            assert all((expanded/m.filename).resolve().is_relative_to(expanded.resolve()) for m in archive.infolist())
            archive.extractall(expanded)
        manifests = [expanded/'kaggle_kit.json']
    assert len(manifests) == 1, 'Chỉ gắn một phiên bản kit, không gắn trùng'
    kit = manifests[0].parent
    metadata = json.loads(manifests[0].read_text())
    assert metadata['format'] == 'datamate-kaggle-kit-v1'
    for relative, expected in metadata['files'].items():
        path = (kit/relative).resolve()
        assert path.is_relative_to(kit.resolve()), 'Đường dẫn kit không hợp lệ'
        with path.open('rb') as handle:
            assert hashlib.file_digest(handle, 'sha256').hexdigest() == expected, relative
    source_hash = metadata['source_archives']['current']['sha256']
    toolkit = bootstrap / 'toolkit' / source_hash[:16]
    if not toolkit.exists():
        shutil.copytree(kit/'current', toolkit)
    workflow = runpy.run_path(str(toolkit/'tools/kaggle_workflow.py'))
    OPTIONS['current_zip_sha256'] = source_hash
    OPTIONS['kit_root'] = str(kit)
    STATUS = workflow['entrypoint'](TASK, **OPTIONS)
    print(json.dumps(STATUS, ensure_ascii=False, indent=2))
except BaseException:
    output = Path(OPTIONS.get('output_root', '/kaggle/working'))
    output.mkdir(parents=True, exist_ok=True)
    (output/'bootstrap_error.log').write_text(traceback.format_exc(), encoding='utf-8')
    raise



<!-- notebook:compare:end -->

Nếu không tìm thấy đúng một file mỗi tên, cell sẽ báo các file đang thấy. Quay lại Add Input chọn đúng version main, bỏ nguồn bị gắn trùng. Không sửa tên file của pilot thành main để vượt kiểm tra.

Kết quả so sánh được in ngay trong cell và ghi rõ:

| File trong `/kaggle/working` của notebook CPU | Nội dung |
|---|---|
| **`comparison.txt`** | Kết luận dễ đọc: điều kiện nào chưa qua, thời gian và speedup |
| **`comparison.csv`** | Bảng Transformers/vLLM để xem hoặc đưa vào báo cáo |
| `comparison.json` | Đầy đủ số liệu, điều kiện so sánh và ID bất đồng |
| `comparison_status.json`, `comparison_run.log` | Trạng thái/lỗi của việc ghép kết quả |
| `comparison_result.zip` | Gói các file so sánh trên để lưu lại |

Ví dụ thao tác sau khi đã chạy GPU:

1. Vào version main của notebook Torch. Mở `torch_main_status.json`: xem số câu và `passed`/`needs_review`. Kit mới nhận cả hai để tạo báo cáo; trạng thái cảnh báo không bị đổi thành passed.
2. Làm tương tự notebook vLLM. Với ZIP cũ có `failed` sau bước `run_main_questions` nhưng metrics/submission vẫn đủ 24 câu, cập nhật kit mới rồi chạy notebook so sánh CPU; không cần chạy lại GPU chỉ để đọc các số đo đã có. ZIP thiếu metrics, thiếu câu hoặc hỏng vẫn không được coi là lượt so sánh sạch.
3. Mở notebook CPU **hai-runtime-compare**, Add Input Output của **hai version ở bước 1–2**. Hai file đầu vào cần có là `torch_main_result.zip` và `vllm_main_result.zip`. Không cần tự giải nén hay chọn từng log con.
4. Chạy cell của `DataMate_Kaggle_Compare.ipynb`. Cell tự đọc số đo trong hai ZIP; nó không nạp model và không xử lý lại 24 câu.
5. Đọc phần chữ in cuối cell hoặc mở `comparison.txt`. Muốn lấy bảng đưa cho nhóm thì dùng `comparison.csv`. File `comparison.json` dành cho xem chi tiết khi có điều kiện chưa đạt hoặc đáp án khác nhau.

Nếu đọc `comparison.txt` thấy “Chưa đủ điều kiện” hoặc một điều kiện chưa đạt, chưa được dùng ratio thời gian để khẳng định vLLM nhanh hơn. Đọc lý do được ghi bằng tiếng Việt và mở log/metrics của lượt tương ứng.

Muốn lưu Output để quay lại sau, dùng Save Version → Save & Run All của notebook CPU này; nó tự chạy lại từ hai Input đã lưu, không chạy lại GPU. Có thể tải trực tiếp `comparison_result.zip` và hai main_result.zip từ Output nếu muốn giữ bản riêng. Không có bước bắt buộc chạy so sánh trên máy cá nhân.

## 7. Đọc kết quả theo thứ tự này

1. Mở **comparison.txt**, xem điều kiện so sánh có hợp lệ không. Khác IDs/dataset/checkpoint/precision/thư viện/prompt đầu, sai backend, thiếu câu hoặc có deadline thì không được kết luận speedup sạch. Báo cáo vẫn giữ tất cả câu và phân loại nguyên nhân; `needs_review` không phải engine bị crash.
2. Xem **Inference giây** và **Tổng run_batch giây**. Speedup = thời gian Torch / thời gian vLLM: >1 là vLLM nhanh hơn trong lượt này, <1 là chậm hơn. Tổng run_batch còn tính cả kiểm hash/load model; **không gồm tải model, cài thư viện hay context preflight**. Không lấy `elapsed_seconds` trong status hoặc thời lượng toàn notebook làm tốc độ inference: các số đó có cả phần chuẩn bị tự động.
3. Xem câu/phút, latency p50/p95, số calls và token output. Sinh ít token, abstain nhiều hoặc làm ít lượt model có thể khiến thời gian ngắn hơn; đó chưa tự là tối ưu tốt.
4. Xem **Token cache hit**: số lấy trực tiếp từ vLLM, không suy từ việc có hai request chạy đồng thời. `null` là chưa có số đo, không phải 0%. Generate đồng thời cao nhất chỉ là overlap ở caller, không phải batch size nội bộ engine.
5. Không có nhãn thì chỉ kết luận timing/agreement. Hai đáp án giống nhau không đồng nghĩa đều đúng. Muốn kết luận nhanh hơn mà không giảm chất lượng cần chấm nhãn TRAIN sau inference; không đưa nhãn vào Input của agent.

Bản kiểm thử mới tách tỷ số thời gian quan sát của toàn pipeline khỏi speedup sạch. Khi có timeout, tỷ số quan sát vẫn được in nhưng `inference_speedup` giữ null. `comparison.json` có `per_question`: thời gian trước lần gọi model đầu, tổng thời gian gọi model, phần thời gian còn lại, khác biệt có deadline hay hai bên trả đáp án khác nhau, action phân kỳ đầu tiên và khác biệt phản hồi tool. Phần ngoài generate gồm chuẩn bị dữ liệu, chờ lock và chạy tool; số đo này không tự phân biệt từng nguyên nhân đó. Không bỏ các câu chậm/lỗi khỏi bảng.

Mặc định vẫn 90 giây/câu. Nếu cần một phép đo với ngân sách khác, có thể đặt `SETTINGS = {'question_timeout_seconds': 180}` ở **cả hai** notebook; đó là lượt thử khác, phải giữ nguyên cùng giá trị ở hai bên. Không đổi budget một bên hoặc chuyển đồng hồ ra sau bước chuẩn bị dữ liệu. Phép đo hiện tại là toàn pipeline với cách điều phối khác nhau; chưa phải phép đo engine trên chuỗi request cố định.

Hai result ZIP còn có thư mục `details/...` chứa `benchmark_metrics.json`, `benchmark_identity.json`, `model_calls.jsonl`, `question_timing.jsonl`, `gpu_samples.jsonl`, `submission.jsonl` và log từng câu. Không cần tự tải từng file để so sánh: notebook CPU đọc chúng trong ZIP. Nếu debug một ID, tìm `id` trong log JSON; vLLM cache count ở metadata generation. GPU/RAM là số lấy mẫu, có thể bỏ lỡ đỉnh ngắn.



Xem huongdankaggle.md để đi tiếp theo thứ tự; tất cả bước chuẩn bị, chạy và so sánh đều trên Kaggle.